# NB_MDP_10 · Bronze (una entidad)

Lo llama `NB_MDP_00_orquestador` con `runMultiple`, una vez por entidad. También se puede correr solo, pegando el JSON de una entidad en `entity_json`.

- Lee la carpeta de la corrida (`Files/landing/<entidad>/<run_id>/`) si el origen es SQL, o la carpeta de OneLake configurada si el origen es FILES.
- Agrega columnas de auditoría (`_run_id`, `_ingested_at`, `_source_file`, `_file_modified_at`) y hace **append** en la tabla bronze. Bronze no transforma: guarda lo que llegó.
- Calcula el watermark nuevo (máximo de la columna o fecha de modificación del archivo) y lo **devuelve**; no escribe en la base de control. El watermark lo avanza `log.usp_END_RUN` solo si este paso terminó bien.
- Nunca corta con excepción: los errores vuelven en el JSON de salida para que el orquestador siga con las otras entidades.

In [ ]:
# Parámetros (celda marcada como "parameters": la pisa el orquestador o el pipeline)
run_id = "manual"
entity_json = "{}"

In [ ]:
import json, re
from datetime import date, datetime
from decimal import Decimal
from pyspark.sql import functions as F

e = json.loads(entity_json)
resultado = {"layer": "bronze", "object_code": e.get("entity_code"), "status": None,
             "rows_read": None, "rows_written": None, "new_watermark": None, "error": None}


def nombre_valido(c):
    """Delta (sin column mapping) no acepta espacios ni ,;{}()= en los nombres de columna."""
    return re.sub(r"[ ,;{}()\n\t=]+", "_", c.strip())


def como_texto(v):
    """Formato del watermark que entiende TRY_CONVERT en SQL (ISO 8601) o un entero."""
    if v is None:
        return None
    if isinstance(v, (datetime, date)):
        return v.isoformat()
    if isinstance(v, Decimal):
        return str(int(v))
    return str(v)


def max_como_texto(df, columna):
    """Máximo de la columna como texto. Las fechas se formatean dentro de Spark, en la zona
    horaria de la sesión, que es la misma con la que Spark interpreta modifiedAfter."""
    tipo = df.schema[columna].dataType.simpleString()
    if tipo.startswith("timestamp"):
        return df.agg(F.date_format(F.max(columna), "yyyy-MM-dd'T'HH:mm:ss.SSSSSS")).first()[0]
    if tipo == "date":
        return df.agg(F.date_format(F.max(columna), "yyyy-MM-dd")).first()[0]
    return como_texto(df.agg(F.max(columna)).first()[0])


def sin_archivos(ex):
    """Spark no puede inferir el esquema cuando no quedó ningún archivo para leer
    (por ejemplo, porque modifiedAfter los filtró a todos)."""
    return "UNABLE_TO_INFER_SCHEMA" in str(ex) or "Unable to infer schema" in str(ex)


def leer_origen(e):
    ruta, fmt = e["read_path"], e["file_format"]
    if fmt == "delta":
        return spark.read.format("delta").load(ruta)
    lector = spark.read.format(fmt)
    if fmt == "csv":
        # Todo como texto: los tipos se definen en silver (meta.ENTITY_COLUMN)
        lector = lector.option("header", "true").option("inferSchema", "false")
    if (e["source_type"] == "FILES" and e["load_type"] == "INCREMENTAL"
            and not e.get("watermark_column") and e.get("watermark_value")):
        # Solo archivos modificados después del último watermark (ISO 8601, zona horaria de la sesión)
        lector = lector.option("modifiedAfter", e["watermark_value"])
    return lector.load(ruta)

In [ ]:
try:
    df = None
    if e.get("copy_failed"):
        resultado["status"] = "SKIPPED"
        resultado["error"] = "La copia de esta corrida falló: no se lee la carpeta porque puede estar incompleta"
    elif not notebookutils.fs.exists(e["read_path"]):
        resultado["status"] = "SKIPPED_NO_DATA"
    else:
        try:
            df = leer_origen(e)
        except Exception as ex:
            if not sin_archivos(ex):
                raise
            df = None
    if resultado["status"] is None and df is None:
        resultado.update(status="SKIPPED_NO_DATA", rows_read=0, rows_written=0)
    elif resultado["status"] is None:
        df = df.toDF(*[nombre_valido(c) for c in df.columns])
        es_archivo = e["file_format"] != "delta"
        if es_archivo:
            df = (df.withColumn("_source_file", F.col("_metadata.file_path"))
                    .withColumn("_file_modified_at", F.col("_metadata.file_modification_time")))

        col_wm = nombre_valido(e["watermark_column"]) if e.get("watermark_column") else None
        if e["source_type"] == "FILES" and e["load_type"] == "INCREMENTAL" and col_wm and e.get("watermark_value"):
            # En SQL el filtro ya viene en la consulta de origen; en FILES con columna se filtra acá
            df = df.where(F.col(col_wm) >= F.lit(e["watermark_value"]).cast(df.schema[col_wm].dataType))

        df = (df.withColumn("_run_id", F.lit(run_id))
                .withColumn("_ingested_at", F.current_timestamp())
                .persist())
        filas = df.count()

        if filas == 0:
            resultado.update(status="SKIPPED_NO_DATA", rows_read=0, rows_written=0)
        else:
            if e["load_type"] == "INCREMENTAL":
                if col_wm:
                    resultado["new_watermark"] = max_como_texto(df, col_wm)
                elif es_archivo:
                    resultado["new_watermark"] = max_como_texto(df, "_file_modified_at")
            (df.write.format("delta").mode("append")
               .option("mergeSchema", "true")          # columnas nuevas en el origen no rompen la carga
               .saveAsTable(e["bronze_table"]))
            resultado.update(status="SUCCEEDED", rows_read=filas, rows_written=filas)
        df.unpersist()
except Exception as ex:
    resultado.update(status="FAILED", error=f"{type(ex).__name__}: {ex}"[:3000])

print(json.dumps(resultado, ensure_ascii=False, indent=2))

In [ ]:
# exit() va en su propia celda y fuera de try/except, como pide la documentación de notebookutils
notebookutils.notebook.exit(json.dumps(resultado, ensure_ascii=False, default=str))